# 05 — Local Validation & Foundation Readiness

**Purpose:** Run all validation gates and confirm the foundation is complete before training.

This notebook:
1. Builds and validates the empty baseline submission.
2. Smoke-tests CER, IoU, reading order, and normalization utilities.
3. Runs the full foundation readiness checklist.
4. Shows where to go next.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── 1. Build empty baseline submission ───────────────────────────
# This proves the submission pipeline works end-to-end.
# An empty submission scores 0 but is correctly formatted.

from src.submission.build_empty_submission import build_empty_submission
csv_path = build_empty_submission()
print(f'\nEmpty submission saved: {csv_path}')

In [ ]:
# ── 2. Validate the submission CSV ───────────────────────────────

from src.submission.validate_submission import validate_submission
from src.utils.paths import get_path, ensure_dir

is_valid, errors = validate_submission(csv_path)

if is_valid:
    print('✅ Submission validation PASSED')
else:
    print(f'❌ Submission validation FAILED ({len(errors)} errors):')
    for e in errors[:10]:
        print(f'  {e}')

# Save submission validation report
report_dir  = ensure_dir(get_path('reports_dir'))
report_path = report_dir / 'submission_validation_report.md'
verdict     = '✅ PASSED' if is_valid else '❌ FAILED'
lines = [f'# Submission Validation Report\n\n**Result: {verdict}**\n\n']
if errors:
    lines.append('## Errors\n\n```\n')
    for e in errors: lines.append(e + '\n')
    lines.append('```\n')
else:
    lines.append('All checks passed. Submission format is correct.\n')
report_path.write_text(''.join(lines), encoding='utf-8')
print(f'Report saved: {report_path}')

In [ ]:
# ── 3. CER utility smoke-test ─────────────────────────────────────

from src.evaluation.cer import compute_cer, compute_batch_cer

# Perfect prediction
cer_perfect = compute_cer('Доброго ранку', 'Доброго ранку')
print(f'CER (perfect match)  : {cer_perfect:.4f}   (expected 0.0)')

# One character wrong
cer_one_err = compute_cer('Добрго ранку', 'Доброго ранку')
print(f'CER (1 char missing) : {cer_one_err:.4f}   (expected ~0.071)')

# Batch CER
preds = ['Доброго ранку', 'Прівіт', '']
gts   = ['Доброго ранку', 'Привіт',  'Привіт']
result = compute_batch_cer(preds, gts, verbose=True)
print(f'\n✅ CER utility OK. Corpus CER = {result["corpus_cer"]:.4f}')

In [ ]:
# ── 4. IoU utility smoke-test ─────────────────────────────────────

from src.evaluation.iou import compute_iou, match_predictions

iou_same  = compute_iou([10,20,100,200], [10,20,100,200])
iou_none  = compute_iou([10,20,100,200], [200,300,400,500])
iou_part  = compute_iou([10,20,100,200], [50,80,150,250])

print(f'IoU (identical boxes): {iou_same:.4f}  (expected 1.0)')
print(f'IoU (no overlap)     : {iou_none:.4f}  (expected 0.0)')
print(f'IoU (partial overlap): {iou_part:.4f}  (expected > 0)')
print('✅ IoU utility OK')

In [ ]:
# ── 5. Reading order smoke-test ───────────────────────────────────

from src.postprocessing.reading_order import sort_regions

scrambled = [
    {'bbox': [400, 100, 800, 140], 'type': 'printed',     'text': 'B'},
    {'bbox': [10,  100, 390, 140], 'type': 'handwritten', 'text': 'A'},
    {'bbox': [10,  200, 390, 240], 'type': 'handwritten', 'text': 'C'},
    {'bbox': [400, 200, 800, 240], 'type': 'printed',     'text': 'D'},
]

ordered = sort_regions(scrambled)
order   = ''.join(r['text'] for r in ordered)
print(f'Input order : {"BACD"}')
print(f'Sorted order: {order}  (expected ABCD)')
assert order == 'ABCD', f'Reading order wrong: {order}'
print('✅ Reading order utility OK')

In [ ]:
# ── 6. Text normalization smoke-test ─────────────────────────────

from src.postprocessing.normalize_text import normalize

tests = [
    ('  Доброго   ранку  ',  'Доброго ранку'),
    ('«Привіт»',             '"Привіт"'),
    ('A\u2013B',             'A-B'),   # en-dash → hyphen
]

for inp, expected in tests:
    result = normalize(inp)
    icon   = '✅' if result == expected else '❌'
    print(f'{icon}  normalize({repr(inp)}) → {repr(result)}')
print('✅ Text normalization OK')

In [ ]:
# ── 7. Foundation readiness checklist ────────────────────────────
# This is the definitive gate. All items must pass before training.

from src.data.foundation_readiness import run_checklist, print_checklist, save_readiness_report

results  = run_checklist()
failures = print_checklist(results)
save_readiness_report(results)

In [ ]:
# ── 8. List generated reports ─────────────────────────────────────

reports_dir = get_path('reports_dir')
reports     = sorted(reports_dir.glob('*.md')) if reports_dir.exists() else []

print(f'\nReports in {reports_dir}:')
for r in reports:
    size = r.stat().st_size
    print(f'  ✅  {r.name:<50}  {size:,} bytes')

if not reports:
    print('  No reports found. Run the earlier steps first.')

## What Comes Next

### If `python -m src.data.foundation_readiness` shows all ✅:

```bash
# Install training packages:
pip install -r requirements-training.txt

# Start detector training (YOLO):
# See configs/detector.yaml for full training command.
# yolo detect train data=data/yolo/data.yaml model=yolov8m.pt imgsz=1024 epochs=50

# Start recognizer training (TrOCR):
# See configs/recognizer.yaml for fine-tuning instructions.
```

### If any check is ❌:
Each failing item in the checklist includes a `FIX:` command to run.
Resolve them one by one, then re-run this notebook.

---

**Remember:**
- Never use `silver` data until the `train` pipeline works end-to-end.
- Never train on `test` images.
- `sample_submission.csv` is a format guide — not training data.
- Final inference must use open-weight models only (no proprietary APIs).